# Dense EWC-coupled MNIST experiment

This notebook reads completed Phase 6 artifacts only. Each Fisher condition follows its own parameter path, so tracking errors use references evaluated at that condition's exact checkpoint. A replica is the statistical unit.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = Path.cwd().parent
if not (REPO_ROOT / "src").is_dir():
    raise RuntimeError("Run this notebook from the repository root or mnist_experiment directory")
sys.path.insert(0, str(REPO_ROOT))

from src.results_analysis import (
    discover_phase6_runs,
    phase6_condition_rows,
    phase6_replica_summaries,
)

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {
    "ema": "#2166ac",
    "ac_only": "#b2182b",
    "full_lfu": "#762a83",
    "periodic_fresh": "#1b7837",
    "ridge_ac_only": "#e08214",
    "ridge_full_lfu": "#008b8b",
}
RUN_ROOT = REPO_ROOT / "cache/mnist_experiment/phase6_runs"
all_runs = discover_phase6_runs(RUN_ROOT)
runs = tuple(run for run in all_runs if run.config.experiment == "mnist_lfu_phase6_convergent_pilot")
if not runs:
    runs = tuple(run for run in all_runs if not run.config.experiment.endswith("_smoke"))
if not runs:
    runs = all_runs
rows = pd.DataFrame(phase6_condition_rows(runs))
summaries = pd.DataFrame(phase6_replica_summaries(rows.to_dict("records")))
print(f"Loaded {len(runs)} completed Phase 6 run(s), {rows['replica_id'].nunique()} replica(s), and {rows['method'].nunique()} methods.")

## Objective and provenance

The code minimizes the mean new-data loss plus the old-to-new EWC odds, $\lambda_t=(1-\pi_t)/\pi_t$, and accepts the optimizer solution directly.

In [ ]:
provenance = pd.DataFrame([
    {
        "run_id": run.run_id,
        "replica_id": run.config.replica_id,
        "pi": run.metrics["adaptation"]["adaptation_weight"],
        "pi_max": run.metrics["adaptation"]["pi_max"],
        "ewc_odds": run.metrics["adaptation"]["effective_ewc_strength"],
        "post_scaling": run.metrics["adaptation"]["post_optimization_scaling"],
        "shared_stream": run.metrics["pairing"]["shared_observation_stream"],
        "path_diverged": run.metrics["pairing"]["path_diverged"],
    }
    for run in runs
])
print(provenance.to_string(index=False))
if rows['replica_id'].nunique() < 2:
    print("\nDescriptive only: fewer than two independent replicas are available.")

## Divergent paths and own-path Fisher tracking

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for run in runs:
    divergence = pd.DataFrame(run.metrics["path_divergence"])
    axes[0].plot(divergence["p"], divergence["maximum_pairwise_parameter_distance"], marker="o", label=run.config.replica_id)
for method, group in rows.groupby("method", sort=False):
    trajectory = group.groupby("p", as_index=False)["relative_frobenius_error"].mean()
    axes[1].plot(trajectory["p"], trajectory["relative_frobenius_error"], marker="o", label=method, color=COLORS[method])
axes[0].set_xlabel("Environmental digit-9 proportion p")
axes[0].set_ylabel("Maximum pairwise parameter distance")
axes[0].set_title("Condition path divergence")
axes[0].legend()
axes[1].set_xlabel("Environmental digit-9 proportion p")
axes[1].set_ylabel("Relative Frobenius error")
axes[1].set_title("Reference evaluated on each condition's path")
axes[1].legend(fontsize=8, ncol=2)
fig.tight_layout()

## Retention and adaptation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.4), sharex=True)
axes = axes.flat
for method, group in rows.groupby("method", sort=False):
    means = group.groupby("p", as_index=False).agg(
        non_nine=("before_non_nine_accuracy", "mean"),
        nine_nll=("before_nine_nll", "mean"),
        non_nine_nll=("before_non_nine_nll", "mean"),
        balanced=("before_balanced_accuracy", "mean"),
    )
    for axis, metric in zip(axes, ["non_nine", "nine_nll", "non_nine_nll", "balanced"]):
        axis.plot(means["p"], means[metric], marker="o", label=method, color=COLORS[method])
for axis, title in zip(axes, ["Digits 0-8 retention", "Digit 9 adaptation (NLL)", "Digits 0-8 NLL", "Balanced accuracy"]):
    axis.set_xlabel("p")
    axis.set_title(title)
axes[0].set_ylabel("Accuracy")
axes[1].set_ylabel("NLL; lower is better")
axes[2].set_ylabel("NLL; lower is better")
axes[3].set_ylabel("Accuracy")
axes[-1].legend(fontsize=8, ncol=2)
fig.tight_layout()

print(summaries.to_string(index=False, float_format=lambda value: f"{value:.6g}"))

## Optimization behavior

The final tessellation point has no outgoing update and is omitted from these panels.

In [ ]:
updates = rows[rows["proposal_displacement_norm"].notna()].copy()
fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.0))
for method, group in updates.groupby("method", sort=False):
    means = group.groupby("p", as_index=False).agg(
        data_loss=("proposal_data_loss_after", "mean"),
        ewc_penalty=("proposal_ewc_penalty", "mean"),
        displacement=("proposal_displacement_norm", "mean"),
    )
    for axis, metric in zip(axes, ["data_loss", "ewc_penalty", "displacement"]):
        axis.plot(means["p"], means[metric], marker="o", label=method, color=COLORS[method])
for axis, title in zip(axes, ["New-data loss after update", "EWC penalty", "Realized optimizer displacement"]):
    axis.set_xlabel("p")
    axis.set_title(title)
axes[0].set_ylabel("Stored scalar metric")
axes[-1].legend(fontsize=8, ncol=2)
fig.tight_layout()

## Interpretation boundary

A controlled or capped $\pi_t$ need not equal the literal sample proportion. It then defines a tempered objective that may lag the instantaneous MLE in exchange for variance reduction, old-task retention, and smaller movements. This notebook reports that trade-off; it does not reinterpret repeated trajectory checkpoints as independent samples.